<a href="https://colab.research.google.com/github/simply-logical/mlbook_ii_notebooks/blob/master/Data_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install datasets pyarrow mlcroissant --quiet

In [ ]:
import matplotlib.pyplot as plt
from abc import ABC, abstractmethod

import re
import os
import tarfile
import requests

import pandas as pd
import seaborn as sns
from mlcroissant import Dataset

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

In [ ]:
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

In [ ]:
import warnings
import logging
from absl import logging as absl_logging

warnings.filterwarnings("ignore")
absl_logging.set_verbosity(absl_logging.ERROR)
logging.getLogger().setLevel(logging.ERROR)

In [ ]:
class DataSource(ABC):
  @abstractmethod
  def load_data(self) -> pd.DataFrame:
    pass

In [ ]:
class EuroparlDataset(DataSource):
  def __init__(self, language_pair: str="pt-en", save_path: str="data/europarl"):

    self.language_pair = language_pair
    self.save_path = save_path
    self.url = f"https://www.statmt.org/europarl/v7/{language_pair}.tgz"
    self.filename = f"{language_pair}.tgz"
    self.full_path = os.path.join(self.save_path, self.filename)

  def _ensure_directory(self):

    if not os.path.exists(self.save_path):
      os.makedirs(self.save_path)

  def _download(self):

    self._ensure_directory()
    if not os.path.exists(self.full_path):
      print(f"Downloading {self.url}...")
      response = requests.get(self.url, stream=True)
      with open(self.full_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
          f.write(chunk)
      print(f"Download of Europarl {self.language_pair} finished.")

  def _extract_and_load(self) -> pd.DataFrame:

      dataframe = pd.DataFrame()

      with tarfile.open(self.full_path, "r:gz") as tar:
        files = tar.getnames()
        for target_file in files:

          print(f" Extracting: {target_file}")
          f = tar.extractfile(target_file)
          content = f.read().decode("utf-8").splitlines()[:1000]
          dataframe[f'{target_file[-2:]}'] = content

      return dataframe

  def load_data(self)-> pd.DataFrame:

    self._download()
    return self._extract_and_load()




In [ ]:
class HelloSimpleAIDataset(DataSource):

  def __init__(self, source="wiki_csai"):
    self.url = "https://huggingface.co/api/datasets/Hello-SimpleAI/HC3/croissant"
    self.source = source

  def load_data(self) -> pd.DataFrame:

    ds = Dataset(jsonld=self.url)
    records_iterator = ds.records(self.source)

    data = []
    print(f"Processing Hello-SimpleAI - {self.source}")
    data = [item for item in records_iterator]

    print(f"Done.Total records: {len(data)}")
    return pd.DataFrame(data)

In [ ]:
class DataCleaner(ABC):
  @abstractmethod
  def clean_data(self, data: pd.DataFrame) -> pd.DataFrame:
    pass

In [ ]:
class EuroparlDataCleaner(DataCleaner):
  def __init__(self, language_pair: str="pt-en"):
    self.language_pair = language_pair

  def clean_data(self, data: pd.DataFrame) -> pd.DataFrame:

    cleaned_dataset = pd.DataFrame()
    for column in data.columns:
      cleaned_dataset[column] = data[column].apply(lambda text: str.lower(str(text)))

    return cleaned_dataset

In [ ]:
class HelloSimpleAIDataCleaner(DataCleaner):
  def __init__(self, source="wiki_csai"):
    self.source = source

  def _clean_text(self, raw_text):
    text = re.sub(r"^\[?b['\"]|['\"]\]?$", "", raw_text)

    text = re.sub(r"\n", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    text = text.lower()

    return text

  def clean_data(self, data: pd.DataFrame) -> pd.DataFrame:

    for column in data.columns:
      data[column] = data[column].apply(lambda text: self._clean_text(str(text)))

    return data


In [ ]:
class DataProcessor(ABC):
  @abstractmethod
  def process_dataframe(self, data: pd.DataFrame) -> pd.DataFrame:
    pass

In [ ]:
class EuroparlDataProcessor(DataProcessor):
  def __init__(self, language_pair: str="pt-en"):
    self.language_pair = language_pair

  def _remove_punctuation(self, text: str) -> str:
    return re.sub(r'[^\w\s]', '', text)

  def process_dataframe(self, df: pd.DataFrame, text_column: str) -> pd.DataFrame:
        print(f'Processing column: {text_column}')
        df['sentence_list'] = df[text_column].apply(sent_tokenize)

        df_sentences = df.explode('sentence_list').reset_index(drop=True)
        df_sentences = df_sentences.rename(columns={'sentence_list': 'sentence'})

        df_sentences = df_sentences[df_sentences['sentence'].str.strip().astype(bool)]

        df_sentences = df_sentences.dropna(subset=['sentence'])

        df_sentences['tokens'] = df_sentences['sentence'].astype(str).apply(self._remove_punctuation).apply(word_tokenize)
        df_sentences['word_count'] = df_sentences['tokens'].apply(len)

        df_sentences['avg_word_len'] = df_sentences['tokens'].apply(
            lambda words: sum(len(w) for w in words) / len(words) if len(words) > 0 else 0
        )

        df_sentences['char_count'] = df_sentences['sentence'].apply(len)

        return df_sentences

In [ ]:
class HelloSimpleAIDataProcessor(DataProcessor):
  def __init__(self, source="wiki_csai"):
    self.source = source

  def _remove_punctuation(self, text: str) -> str:
    return re.sub(r'[^\w\s]', '', text)

  def process_dataframe(self, df: pd.DataFrame, text_column: str) -> pd.DataFrame:
    print(f'Processing column: {text_column}')

    df['sentence_list'] = df[text_column].apply(sent_tokenize)

    df_sentences = df.explode('sentence_list').reset_index(drop=True)
    df_sentences = df_sentences.rename(columns={'sentence_list': 'sentence'})

    df_sentences = df_sentences[df_sentences['sentence'].str.strip().astype(bool)]

    df_sentences = df_sentences.dropna(subset=['sentence'])

    df_sentences['tokens'] = df_sentences['sentence'].astype(str).apply(self._remove_punctuation).apply(word_tokenize)
    df_sentences['word_count'] = df_sentences['tokens'].apply(len)

    df_sentences['avg_word_len'] = df_sentences['tokens'].apply(
            lambda words: sum(len(w) for w in words) / len(words) if len(words) > 0 else 0
    )

    df_sentences['char_count'] = df_sentences['sentence'].apply(len)

    return df_sentences

In [ ]:
class DataVisualizer:
    @staticmethod
    def plot_bar_chart(df: pd.DataFrame, title: str):
      plt.figure(figsize=(15, 6))
      ax = sns.barplot(data=df, color="#3498db")
      for container in ax.containers:
          ax.bar_label(container, fmt='%.2f', padding=3)

      plt.title(title)
      plt.grid(True, axis='y', linestyle='--', alpha=0.7)

      sns.despine()

      plt.show()
      plt.close()

    @staticmethod
    def plot_histogram(df1:  pd.DataFrame, df2:  pd.DataFrame, label1: str, label2: str, title: str, column: str='word_count'):
      plt.figure(figsize=(14.625,6))

      sns.histplot(
          df1[column],
          kde=False,
          color="#3498db",
          label=label1,
          alpha=0.6,
          edgecolor="white",
          shrink=0.8,
          linewidth=0.25
      )

      plt.title(title)
      plt.xlabel(column)
      plt.ylabel("Frequency")
      plt.legend()

      sns.despine()
      plt.grid(axis='y', alpha=0.3)
      plt.show()
      plt.close()

      plt.figure(figsize=(14.625,6))

      sns.histplot(
          df2[column],
          kde=False,
          color="#e67e22",
          label=label2,
          alpha=0.6,
          edgecolor="white",
          shrink=0.8,
          linewidth=0.25
      )

      plt.title(title)
      plt.xlabel(column)
      plt.ylabel("Frequency")
      plt.legend()

      sns.despine()
      plt.grid(axis='y', alpha=0.3)
      plt.show()
      plt.close()

    @staticmethod
    def plot_scatter_plot(df1: pd.DataFrame, df2: pd.DataFrame, title: str, label1: str, label2: str):
        plt.figure(figsize=(12.165, 6))


        sns.scatterplot(
            data=df1,
            x='char_count',
            y='avg_word_len',
            label=label1,
            alpha=0.7,
            edgecolor=None,
            color="#3498db"
        )


        sns.scatterplot(
            data=df2,
            x='char_count',
            y='avg_word_len',
            label=label2,
            alpha=0.3,
            edgecolor=None,
            color="#e67e22"
        )

        plt.grid(axis='y', alpha=0.3)
        plt.title(title, fontsize=16, pad=20)
        plt.xlabel("Sentence Length", fontsize=12)
        plt.ylabel("Average Word Length", fontsize=12)


        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

        sns.despine()
        plt.tight_layout()
        plt.show()

In [ ]:
data = HelloSimpleAIDataset().load_data()
data_cleaner = HelloSimpleAIDataCleaner()
cleaned_data = data_cleaner.clean_data(data)
data_processor = HelloSimpleAIDataProcessor()
processed_data = data_processor.process_dataframe(cleaned_data, 'wiki_csai/human_answers')
processed_data

In [ ]:
def on_click(b):
    clear_output(wait=True)
    source_name = dropdown.value

    if source_name == "Europarl":
      language = lang_dropdown.value
      language_pair=f"{lang_dropdown.value}-en" if lang_dropdown.value else "pt-en"
      source=EuroparlDataset(language_pair=language_pair)
      data = source.load_data()
      data_cleaner = EuroparlDataCleaner()
      cleaned_data = data_cleaner.clean_data(data)

      data_processor = EuroparlDataProcessor(language_pair=language_pair)
      processed_data = data_processor.process_dataframe(cleaned_data, language)
      processed_data_en = data_processor.process_dataframe(cleaned_data, 'en')

      sentence_len = pd.DataFrame({
        'en': [processed_data_en['word_count'].mean()],
        language: [processed_data['word_count'].mean()]
      })
      DataVisualizer.plot_bar_chart(sentence_len, f" Average sentence length - {dropdown.value}")
      DataVisualizer.plot_histogram(processed_data_en, processed_data, "en", language, f"Sentence length distribution - {dropdown.value}")

      DataVisualizer.plot_scatter_plot(processed_data_en, processed_data, f"Sentence length scatter plot - {dropdown.value}", "en", language)
    else:
      dataset_source = source_dropdown.value
      source=HelloSimpleAIDataset(source=dataset_source)
      data = source.load_data()
      data_cleaner = HelloSimpleAIDataCleaner()
      cleaned_data = data_cleaner.clean_data(data)

      data_processor = HelloSimpleAIDataProcessor()
      processed_data_human = data_processor.process_dataframe(cleaned_data, f'{dataset_source}/human_answers')
      processed_data_gpt = data_processor.process_dataframe(cleaned_data, f'{dataset_source}/chatgpt_answers')

      sentence_len = pd.DataFrame({
        'human': [processed_data_human['word_count'].mean()],
        'gpt': [processed_data_gpt['word_count'].mean()]
      })
      DataVisualizer.plot_bar_chart(sentence_len, f" Average sentence length - {dataset_source}")
      DataVisualizer.plot_histogram(processed_data_human, processed_data_gpt, "human", "gpt", f"Sentence length distribution - {dataset_source}")

      DataVisualizer.plot_scatter_plot(processed_data_human, processed_data_gpt, f"Sentence length scatter plot - {dataset_source}", "human", "gpt")


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

sources = {
    "HelloSimple-AI":  HelloSimpleAIDataset(),
    "Europarl":  EuroparlDataset(),
}


dropdown = widgets.Dropdown(
    options=sources.keys(),
    description='Source:',
    layout=widgets.Layout(width='auto')
)


lang_dropdown = widgets.Dropdown(
    options=['pt', 'es', 'fr', 'da', 'fi','it', 'el', 'et', 'nl'],
    description='Language:',
    layout=widgets.Layout(width='auto', display='none')
)

source_dropdown = widgets.Dropdown(
    options=['wiki_csai', 'all_splits', 'all', 'finance_splits', 'finance', 'medicine_splits', 'medicine', 'open_qa_splits', 'open_qa', 'reddit_eli5_splits', 'reddit_eli5', 'wiki_csai_splits'],
    description='Source:',
    layout=widgets.Layout(width='auto', display='flex')
)

button = widgets.Button(
    description="Plot",
    button_style='info',
    layout=widgets.Layout(width='auto', height='40px')
)

output = widgets.Output()


def on_source_change(change):
    if change['new'] == "Europarl":
        lang_dropdown.layout.display = 'flex'
    else:
        lang_dropdown.layout.display = 'none'

    if change['new'] == "HelloSimple-AI":
        source_dropdown.layout.display = 'flex'
    else:
        source_dropdown.layout.display = 'none'

dropdown.observe(on_source_change, names='value')

button.on_click(on_click)

grid = widgets.GridspecLayout(4, 1, grid_gap='10px')
grid[0, 0] = dropdown
grid[1, 0] = source_dropdown
grid[2, 0] = lang_dropdown
grid[3, 0] = button

grid.layout.width = '400px'
grid.layout.margin = '20px'

display(grid, output)